# 11 — Week 4: N-HiTS, PatchTST, DLinear, NLinear (+ univariate GRU) with three seeds, validation only

All networks use the same protocol as the Week 3 LSTM/GRU (`src/models/nf_models.py`): the same usable origins,
60-minute input window, local scaling, masked MSE on h = 1, 5, 15, Adam 1e-3, batch 512, early stopping on mean
validation MAE (patience 3). Epoch cap 30 for all (provisional; a run whose best epoch equals the cap is flagged).
Inputs are univariate (power window only). `gru_uni` = the Week 3 GRU architecture with the same univariate input,
so that architecture and input channels can be separated in the comparison.
Devices from notebook 10: N-HiTS, DLinear, NLinear on CPU; PatchTST and GRU on MPS.

Cell 2 runs the whole pipeline on synthetic data first (dry run, deleted afterwards). Cell 3 is resumable:
finished runs (`metrics.json` exists) are skipped. **The test split is not scored here.**

In [1]:
# Cell 1: setup
import sys, json, time, shutil, logging, warnings, traceback
from pathlib import Path
import numpy as np
import polars as pl
import torch
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
for name in ("pytorch_lightning", "lightning", "lightning.pytorch", "lightning_fabric"):
    logging.getLogger(name).setLevel(logging.ERROR)

from models import rnn as R
from models import nf_models as NF
from evaluation import metrics as M
from tracking import log_run

GROUP = "week4_nf"
MODELS = ["nhits", "patchtst", "dlinear", "nlinear", "gru_uni"]
DEVICE = {"nhits": "cpu", "patchtst": "mps", "dlinear": "cpu", "nlinear": "cpu", "gru_uni": "mps"}
SEEDS = [0, 1, 2]
CFG = {"lr": 1e-3, "batch": 512, "epochs": 30, "patience": 3, "grad_clip": 1.0, "weight_decay": 0.0,
       "scaling": "local", "calendar": False, "covariates": False,
       "dataset": "data/processed/model_dataset.parquet", "dataset_meta": "data/processed/model_dataset_meta.json"}
REP = ROOT / "reports" / "week4"
REP.mkdir(parents=True, exist_ok=True)
DATA_CFG = {"scaling": "local", "calendar": False, "covariates": False}
print("torch", torch.__version__, "| mps", torch.backends.mps.is_available(), "| models", MODELS, "| seeds", SEEDS)


def arch_name(name):
    return "gru" if name == "gru_uni" else name


def run_one(md, meta, data, name, seed, group, extra_cfg=None, tracking_mode="auto", log=print):
    """Train one model/seed, score validation with the shared metrics, write files and the tracking record."""
    run = f"{name}_s{seed}"
    out = ROOT / "experiments" / group / run
    if (out / "metrics.json").exists():
        log(f"[{time.strftime('%H:%M:%S')}] {run}: done earlier, skipped")
        return json.loads((out / "metrics.json").read_text())
    out.mkdir(parents=True, exist_ok=True)
    cfg = {**CFG, **(extra_cfg or {}), "model": arch_name(name), "name": run, "group": group, "seed": seed,
           "device": DEVICE[name], "family": "nf" if name != "gru_uni" else "rnn_univariate"}
    (out / "config.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    log(f"[{time.strftime('%H:%M:%S')}] {run}: training on {cfg['device']}")
    model, d, hist, info = NF.train(md, cfg, log=log, data=data)
    pl.DataFrame(hist).write_csv(out / "history.csv")
    torch.save(model.state_dict(), out / "model.pt")
    va = d.origins("val")
    t0 = time.time()
    pv = NF.predict(model, d, va, NF.pick_device(cfg["device"]))
    infer_s = time.time() - t0
    L24 = md["L24_kW"].to_numpy().astype(float)
    mod5 = md["mod5"].to_numpy().astype(int)
    ep = md["minute_epoch"].to_numpy()
    rows, metrics, parts = [], {}, []
    for j, h in enumerate(R.HORIZONS):
        t = d.idx[h]["val"]
        yhat = pv[np.searchsorted(va, t), j]
        y, p0 = d.P[t + h], d.P[t]
        ramp = md[f"ramp_h{h}"].to_numpy().astype(float)[t]
        r = M.evaluate(y, yhat, p0, l24=L24[t], ramp=ramp, thr=meta["thresholds"][str(h)]["rel_p95"],
                       mod5_target=mod5[t + h] if h == 1 else None, dm_lag=60)
        rows.append({"h": h, **r})
        metrics.update({f"h{h}/{k}": v for k, v in r.items() if isinstance(v, (int, float))})
        parts.append(pl.DataFrame({"minute_epoch": ep[t], "h": np.full(len(t), h, np.int16), "y": y, "p0": p0,
                                   "yhat": yhat}))
    pl.concat(parts).write_parquet(out / "predictions_val.parquet")
    info["cap_hit"] = bool(info["best_epoch"] == int(cfg["epochs"]))
    metrics.update({"n_params": info["n_params"], "train_s": info["train_s"], "infer_s_val": infer_s,
                    "infer_ms_per_sample": 1000 * infer_s / max(len(va), 1), "best_epoch": info["best_epoch"],
                    "epochs_run": info["epochs_run"], "cap_hit": info["cap_hit"]})
    res = {"info": info, "val": rows, "metrics": metrics}
    log_run(ROOT, run, group, {**cfg, **{f"data_{k}": v for k, v in info.items()}}, metrics,
            files=[out / "config.yaml"], tables={"val_metrics": pl.DataFrame(rows)}, mode=tracking_mode)
    (out / "metrics.json").write_text(json.dumps(res, indent=1, default=str))   # written last = run complete
    log(f"[{time.strftime('%H:%M:%S')}] {run}: params {info['n_params']:,} | train {info['train_s']:.0f} s | "
        f"best epoch {info['best_epoch']}/{info['epochs_run']}{' (CAP HIT)' if info['cap_hit'] else ''} | val MAE " +
        " / ".join(f"{r_['MAE']:.4f}" for r_ in rows) + " kW | skill " +
        " / ".join(f"{r_['skill_MAE']:+.3f}" for r_ in rows))
    return res


torch 2.14.1 | mps True | models ['nhits', 'patchtst', 'dlinear', 'nlinear', 'gru_uni'] | seeds [0, 1, 2]


In [2]:
# Cell 2: DRY RUN of the whole pipeline on SYNTHETIC data (2 epochs, small samples, no tracking; deleted after)
def synth(n=30000, seed=0):
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    x = np.zeros(n)
    e = rng.normal(0, 0.4, n)
    for i in range(1, n):
        x[i] = 0.9 * x[i - 1] + e[i]
    P = 30 + np.cumsum(rng.normal(0, 0.01, n)) + x - 0.3 * (t % 5 == 0)
    valid = np.ones(n, bool)
    for g0 in rng.choice(np.arange(500, n - 500), 6, replace=False):
        valid[g0:g0 + 150] = False
    i_tr, i_va = int(0.7 * n), int(0.85 * n)
    split = np.where(t < i_tr, "train", np.where(t < i_va, "val", "test"))
    c = np.concatenate([[0], np.cumsum(~valid)])
    win_ok = np.zeros(n, bool)
    win_ok[R.INPUT_LEN - 1:] = (c[R.INPUT_LEN:] - c[:n - R.INPUT_LEN + 1]) == 0
    Pv = np.where(valid, P, np.nan)
    L24 = pl.Series(Pv).fill_nan(None).rolling_mean(1440, min_samples=720).to_numpy()
    cols = {"minute_epoch": t * 60, "P_kW": Pv, "valid": valid, "split": split, "mod5": (t % 5).astype(np.int64),
            "hour_sin": np.sin(2 * np.pi * (t // 60 % 24) / 24), "hour_cos": np.cos(2 * np.pi * (t // 60 % 24) / 24),
            "L24_kW": L24}
    thr = {}
    for h in R.HORIZONS:
        ok = win_ok.copy()
        ok[n - h:] = False
        ok[:n - h] &= valid[h:]
        sp = np.full(n, None, dtype=object)
        tt = np.arange(n - h)
        sp[:n - h] = np.where(tt + h < i_tr, "train", np.where((tt >= i_tr) & (tt + h < i_va), "val",
                              np.where(tt >= i_va, "test", None)))
        ok &= sp != None  # noqa: E711
        rel = np.full(n, np.nan)
        rel[:n - h] = np.abs(P[h:] - P[:n - h]) / L24[:n - h]
        trm = ok & (sp == "train") & np.isfinite(rel)
        thr[str(h)] = {"rel_p95": float(np.quantile(rel[trm], 0.95))}
        ramp = np.where(ok & np.isfinite(rel), (rel > thr[str(h)]["rel_p95"]).astype(float), np.nan)
        cols[f"ok_h{h}"], cols[f"split_h{h}"], cols[f"ramp_h{h}"] = ok, sp.tolist(), ramp
    df = pl.DataFrame(cols).with_columns(pl.col("P_kW").fill_nan(None), pl.col("L24_kW").fill_nan(None))
    return df, {"thresholds": thr}

DRY = "_dryrun_week4"
sm, smeta = synth()
sdata = R.Data(sm, DATA_CFG)
dry_ok = True
try:
    for name in MODELS:
        res = run_one(sm, smeta, sdata, name, 0, DRY, extra_cfg={"epochs": 2, "max_train_samples": 5000},
                      tracking_mode="disabled", log=lambda s: None)
        out = ROOT / "experiments" / DRY / f"{name}_s0"
        files = ["config.yaml", "history.csv", "model.pt", "predictions_val.parquet", "metrics.json"]
        missing = [f for f in files if not (out / f).exists()]
        pr = pl.read_parquet(out / "predictions_val.parquet")
        finite = all(np.isfinite(r_["MAE"]) and np.isfinite(r_["skill_MAE"]) for r_ in res["val"])
        good = not missing and finite and pr.height > 0 and pr["yhat"].is_finite().all()
        dry_ok &= bool(good)
        print(f"{name:9s} files ok: {not missing} | prediction rows {pr.height:,} | metrics finite: {finite} | "
              f"skill h1/h5/h15: " + " / ".join(f"{r_['skill_MAE']:+.3f}" for r_ in res["val"]) +
              f" | ramp F1 h15 {res['val'][2]['ramp_F1']:.3f}")
except Exception:
    dry_ok = False
    traceback.print_exc()
finally:
    shutil.rmtree(ROOT / "experiments" / DRY, ignore_errors=True)
    shutil.rmtree(ROOT / "experiments" / "runs" / DRY, ignore_errors=True)
print("\nDRY RUN PASSED (pipeline works end to end; skills on 2 epochs are not meaningful)" if dry_ok
      else "\nDRY RUN FAILED: do not run cell 3, send this output")


2026-10-09 14:55:07,571	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-10-09 14:55:08,167	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
Seed set to 0
Seed set to 0


nhits     files ok: True | prediction rows 12,831 | metrics finite: True | skill h1/h5/h15: +0.026 / +0.081 / +0.160 | ramp F1 h15 0.000


Seed set to 0
Seed set to 0


patchtst  files ok: True | prediction rows 12,831 | metrics finite: True | skill h1/h5/h15: +0.071 / +0.088 / +0.148 | ramp F1 h15 0.048
dlinear   files ok: True | prediction rows 12,831 | metrics finite: True | skill h1/h5/h15: -0.126 / +0.048 / +0.136 | ramp F1 h15 0.010
nlinear   files ok: True | prediction rows 12,831 | metrics finite: True | skill h1/h5/h15: -0.256 / +0.015 / +0.133 | ramp F1 h15 0.000
gru_uni   files ok: True | prediction rows 12,831 | metrics finite: True | skill h1/h5/h15: +0.018 / +0.044 / +0.072 | ramp F1 h15 0.000

DRY RUN PASSED (pipeline works end to end; skills on 2 epochs are not meaningful)


In [3]:
# Cell 3: REAL runs, validation only (resumable). Expected total about 45-60 min; W&B mode "auto".
assert dry_ok, "run cell 2 first and make sure the dry run passed"
md = pl.read_parquet(ROOT / CFG["dataset"])
meta = json.loads((ROOT / CFG["dataset_meta"]).read_text())
data = R.Data(md, DATA_CFG)
print(f"rows {md.height:,} | train origins {len(data.origins('train')):,} | val origins {len(data.origins('val')):,}")
T = time.time()
results = {}
for name in MODELS:
    for seed in SEEDS:
        try:
            results[(name, seed)] = run_one(md, meta, data, name, seed, GROUP)
        except Exception:
            print(f"!!! {name}_s{seed} failed:")
            traceback.print_exc(limit=5)
print(f"\nall runs handled in {(time.time() - T) / 60:.1f} min | finished: {len(results)} of {len(MODELS) * len(SEEDS)}")


Seed set to 0


rows 303,840 | train origins 196,700 | val origins 42,575
[14:55:19] nhits_s0: training on cpu
  epoch   1 | train loss 1.88721 | val MAE 0.2606 / 0.4006 / 0.5736 kW | 4 s
  epoch   2 | train loss 1.84814 | val MAE 0.2599 / 0.4011 / 0.5701 kW | 7 s
  epoch   3 | train loss 1.83291 | val MAE 0.2658 / 0.3988 / 0.5698 kW | 11 s
  epoch   4 | train loss 1.82858 | val MAE 0.2629 / 0.4160 / 0.5783 kW | 15 s
  epoch   5 | train loss 1.81504 | val MAE 0.2617 / 0.4022 / 0.5700 kW | 20 s


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/aryaneuse/.netrc.
Seed set to 1


[14:55:48] nhits_s0: params 676,301 | train 20 s | best epoch 2/5 | val MAE 0.2599 / 0.4011 / 0.5701 kW | skill +0.093 / +0.067 / +0.058
[14:55:48] nhits_s1: training on cpu
  epoch   1 | train loss 1.89168 | val MAE 0.2635 / 0.3991 / 0.5682 kW | 5 s
  epoch   2 | train loss 1.84885 | val MAE 0.2722 / 0.4081 / 0.5860 kW | 10 s
  epoch   3 | train loss 1.83865 | val MAE 0.2614 / 0.4028 / 0.5729 kW | 16 s
  epoch   4 | train loss 1.82689 | val MAE 0.2668 / 0.4060 / 0.5688 kW | 21 s


Seed set to 2


...

[15:08:45] dlinear_s2: params 1,830 | train 1 s | best epoch 2/5 | val MAE 0.2665 / 0.4208 / 0.5868 kW | skill +0.070 / +0.021 / +0.030
[15:08:45] nlinear_s0: training on cpu
  epoch   1 | train loss 2.08493 | val MAE 0.2735 / 0.4227 / 0.5916 kW | 0 s
  epoch   2 | train loss 1.96032 | val MAE 0.2694 / 0.4223 / 0.5869 kW | 0 s
  epoch   3 | train loss 1.94847 | val MAE 0.2671 / 0.4149 / 0.5858 kW | 1 s
  epoch   4 | train loss 1.94585 | val MAE 0.2699 / 0.4213 / 0.5875 kW | 1 s
  epoch   5 | train loss 1.94562 | val MAE 0.2671 / 0.4201 / 0.5889 kW | 1 s
  epoch   6 | train loss 1.94492 | val MAE 0.2680 / 0.4156 / 0.5877 kW | 1 s


Seed set to 1


[15:08:57] nlinear_s0: params 915 | train 1 s | best epoch 3/6 | val MAE 0.2671 / 0.4149 / 0.5858 kW | skill +0.068 / +0.035 / +0.032
[15:08:57] nlinear_s1: training on cpu
  epoch   1 | train loss 2.11615 | val MAE 0.2747 / 0.4186 / 0.5918 kW | 0 s
  epoch   2 | train loss 1.96092 | val MAE 0.2918 / 0.4383 / 0.5951 kW | 0 s
  epoch   3 | train loss 1.95013 | val MAE 0.2677 / 0.4175 / 0.5886 kW | 1 s
  epoch   4 | train loss 1.94738 | val MAE 0.2818 / 0.4216 / 0.5923 kW | 1 s
  epoch   5 | train loss 1.94664 | val MAE 0.2658 / 0.4156 / 0.5864 kW | 1 s
  epoch   6 | train loss 1.94623 | val MAE 0.2702 / 0.4164 / 0.5873 kW | 1 s
  epoch   7 | train loss 1.94661 | val MAE 0.2685 / 0.4140 / 0.5898 kW | 2 s
  epoch   8 | train loss 1.94364 | val MAE 0.2695 / 0.4153 / 0.5861 kW | 2 s


Seed set to 2


[15:09:18] nlinear_s1: params 915 | train 2 s | best epoch 5/8 | val MAE 0.2658 / 0.4156 / 0.5864 kW | skill +0.072 / +0.033 / +0.031
[15:09:18] nlinear_s2: training on cpu
  epoch   1 | train loss 2.05271 | val MAE 0.2742 / 0.4230 / 0.5922 kW | 0 s
  epoch   2 | train loss 1.95995 | val MAE 0.2669 / 0.4223 / 0.5881 kW | 0 s
  epoch   3 | train loss 1.95046 | val MAE 0.2716 / 0.4168 / 0.5892 kW | 1 s
  epoch   4 | train loss 1.94563 | val MAE 0.2796 / 0.4157 / 0.5866 kW | 1 s
  epoch   5 | train loss 1.94476 | val MAE 0.2698 / 0.4170 / 0.5922 kW | 1 s
[15:09:32] nlinear_s2: params 915 | train 1 s | best epoch 2/5 | val MAE 0.2669 / 0.4223 / 0.5881 kW | skill +0.069 / +0.018 / +0.028
[15:09:32] gru_uni_s0: training on mps
  epoch   1 | train loss 1.94266 | val MAE 0.2752 / 0.4071 / 0.5794 kW | 37 s
  epoch   2 | train loss 1.88743 | val MAE 0.2729 / 0.4053 / 0.5760 kW | 73 s
  epoch   3 | train loss 1.86571 | val MAE 0.2757 / 0.4081 / 0.5772 kW | 109 s
  epoch   4 | train loss 1.85011 |

In [4]:
# Cell 4: per-run table and mean +- sd over seeds (validation)
rows = []
for name in MODELS:
    for seed in SEEDS:
        p = ROOT / "experiments" / GROUP / f"{name}_s{seed}" / "metrics.json"
        if not p.exists():
            print("missing:", p.parent.name)
            continue
        r = json.loads(p.read_text())
        for v in r["val"]:
            rows.append({"model": name, "seed": seed, **{k: v.get(k) for k in
                         ["h", "MAE", "RMSE", "sMAPE", "R2", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal",
                          "ramp_precision", "ramp_recall", "ramp_F1", "MAE_no_mod0_target", "DM_p"]},
                         "n_params": r["info"]["n_params"], "train_s": r["info"]["train_s"],
                         "best_epoch": r["info"]["best_epoch"], "epochs_run": r["info"]["epochs_run"],
                         "cap_hit": r["info"].get("cap_hit", False), "device": r["info"]["device"],
                         "infer_ms_per_sample": r["metrics"]["infer_ms_per_sample"]})
runs = pl.DataFrame(rows)
runs.write_csv(REP / "nf_val_runs.csv")
summ = (runs.group_by("model", "h").agg(
            pl.len().alias("n_seeds"), pl.col("MAE").mean().alias("MAE_mean"), pl.col("MAE").std().alias("MAE_sd"),
            *[pl.col(c).mean() for c in ["RMSE", "sMAPE", "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal",
                                         "ramp_recall", "ramp_F1", "MAE_no_mod0_target"]],
            pl.col("DM_p").max().alias("DM_p_max"), pl.col("n_params").first(), pl.col("train_s").mean(),
            pl.col("best_epoch").mean(), pl.col("cap_hit").any(), pl.col("device").first(),
            pl.col("infer_ms_per_sample").mean())
          .sort("h", "MAE_mean"))
summ.write_csv(REP / "nf_val_summary.csv")
pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(20); pl.Config.set_tbl_width_chars(250)
print(summ.select("h", "model", "n_seeds", pl.col("MAE_mean").round(4), pl.col("MAE_sd").round(4),
                  pl.col("skill_MAE").round(3), pl.col("MAE_ramp").round(3), pl.col("MAE_normal").round(4),
                  pl.col("ramp_F1").round(3), pl.col("DM_p_max"), "n_params", pl.col("train_s").round(0),
                  pl.col("best_epoch").round(1), "cap_hit", "device"))
print("\nruns with best epoch at the cap:", runs.filter(pl.col("cap_hit")).select("model", "seed").unique().to_dicts())


shape: (15, 15)
┌─────┬──────────┬─────────┬──────────┬────────┬───────────┬──────────┬────────────┬─────────┬────────────┬──────────┬─────────┬────────────┬─────────┬────────┐
│ h   ┆ model    ┆ n_seeds ┆ MAE_mean ┆ MAE_sd ┆ skill_MAE ┆ MAE_ramp ┆ MAE_normal ┆ ramp_F1 ┆ DM_p_max   ┆ n_params ┆ train_s ┆ best_epoch ┆ cap_hit ┆ device │
│ --- ┆ ---      ┆ ---     ┆ ---      ┆ ---    ┆ ---       ┆ ---      ┆ ---        ┆ ---     ┆ ---        ┆ ---      ┆ ---     ┆ ---        ┆ ---     ┆ ---    │
│ i64 ┆ str      ┆ u32     ┆ f64      ┆ f64    ┆ f64       ┆ f64      ┆ f64        ┆ f64     ┆ f64        ┆ i64      ┆ f64     ┆ f64        ┆ bool    ┆ str    │
╞═════╪══════════╪═════════╪══════════╪════════╪═══════════╪══════════╪════════════╪═════════╪════════════╪══════════╪═════════╪════════════╪═════════╪════════╡
│ 1   ┆ patchtst ┆ 3       ┆ 0.2543   ┆ 0.0015 ┆ 0.112     ┆ 1.324    ┆ 0.2056     ┆ 0.175   ┆ 2.7085e-37 ┆ 75215    ┆ 203.0   ┆ 12.3       ┆ false   ┆ mps    │
│ 1   ┆ nhits    ┆

In [5]:
# Cell 5: merge with the Week 3 benchmark (validation) and print the ranking per horizon
w3 = pl.read_csv(ROOT / "reports" / "week3" / "benchmark_val.csv")
nf = (summ.with_columns(pl.lit("deep").alias("family"), pl.lit(False).alias("covariates"))
          .select("h", "family", "model", "covariates", "n_seeds", "MAE_mean", "MAE_sd", "RMSE", "sMAPE",
                  "R2_delta", "skill_MAE", "MAE_ramp", "MAE_normal", "ramp_F1", pl.col("DM_p_max").alias("DM_p")))
w3c = w3.select(nf.columns).with_columns([pl.col(c).cast(nf.schema[c]) for c in nf.columns])
bench = pl.concat([w3c, nf.with_columns([pl.col(c).cast(nf.schema[c]) for c in nf.columns])]).sort("h", "MAE_mean")
bench.write_csv(REP / "benchmark_val_week4_nf.csv")
for h in R.HORIZONS:
    print(f"\n=== h = {h} min (validation; sorted by MAE) ===")
    print(bench.filter(pl.col("h") == h).select("family", "model", "covariates", "n_seeds",
          pl.col("MAE_mean").round(4), pl.col("MAE_sd").round(4), pl.col("skill_MAE").round(3),
          pl.col("MAE_ramp").round(3), pl.col("MAE_normal").round(4), pl.col("ramp_F1").round(3)))
print("\nwritten:", REP / "nf_val_runs.csv", "|", REP / "nf_val_summary.csv", "|", REP / "benchmark_val_week4_nf.csv")
print("note: DM_p for the new models = largest p-value over the three seeds (conservative), vs persistence")



=== h = 1 min (validation; sorted by MAE) ===
shape: (18, 10)
┌─────────────┬───────────────────┬────────────┬─────────┬──────────┬────────┬───────────┬──────────┬────────────┬─────────┐
│ family      ┆ model             ┆ covariates ┆ n_seeds ┆ MAE_mean ┆ MAE_sd ┆ skill_MAE ┆ MAE_ramp ┆ MAE_normal ┆ ramp_F1 │
│ ---         ┆ ---               ┆ ---        ┆ ---     ┆ ---      ┆ ---    ┆ ---       ┆ ---      ┆ ---        ┆ ---     │
│ str         ┆ str               ┆ bool       ┆ u32     ┆ f64      ┆ f64    ┆ f64       ┆ f64      ┆ f64        ┆ f64     │
╞═════════════╪═══════════════════╪════════════╪═════════╪══════════╪════════╪═══════════╪══════════╪════════════╪═════════╡
│ ML          ┆ lightgbm_cov      ┆ true       ┆ 3       ┆ 0.2464   ┆ 0.0003 ┆ 0.14      ┆ 1.276    ┆ 0.1998     ┆ 0.266   │
│ ML          ┆ lightgbm          ┆ false      ┆ 3       ┆ 0.2491   ┆ 0.0001 ┆ 0.131     ┆ 1.341    ┆ 0.1993     ┆ 0.191   │
│ deep        ┆ patchtst          ┆ false      ┆ 3       ┆ 0.2

In [7]:
# Cell 6: export outputs (save the notebook first with Cmd+S, then run this cell)
%run ../src/tools/nb_outputs_to_md.py 11_week4_nf_models.ipynb


written 11_week4_nf_models_outputs.md | code cells 6 | figures 0
